# Pré-processamento de Dados

**Inteligência Artificial · Prof. Cristiano Rodrigues · PUC Minas**

Este notebook tem três partes:

1. **Qualidade dos dados** (demonstração): auditoria dos quatro pilares, diagnóstico do mecanismo de ausência e imputação, com o dataset `df_clientes`.
2. **Prática guiada** (vocês fazem sozinhos): escala, codificação categórica e `Pipeline`/`ColumnTransformer`, com feedback automático a cada etapa.
3. **Aplicação ao TP1**: o mesmo fluxo aplicado ao `koi_data.csv` de verdade — a ponte direta para o trabalho prático.

> Se esta não é a primeira vez que vocês usam um notebook neste curso, podem pular direto para a Parte 1. Se ainda têm dúvida sobre como rodar células, leiam a célula de aquecimento abaixo primeiro.

## Aquecimento: como usar este notebook

Um notebook é uma sequência de células. Células de texto (como esta) só explicam o que vem a seguir; células de código são onde vocês rodam Python de verdade.

Para rodar uma célula: cliquem nela e apertem **Shift+Enter**. Isso executa o código e move para a próxima célula. Rodem a célula abaixo para testar.

In [1]:
meu_nome = "Lucca"
print(f"Oi, {meu_nome}! Bem-vindo ao notebook da Aula 12.")

Oi, Lucca! Bem-vindo ao notebook da Aula 12.


Editem a célula acima, troquem o texto entre aspas pelo seu nome, e rodem de novo (Shift+Enter). O resultado deve mudar.

**Regra de ouro do notebook**: as células devem ser executadas **em ordem**, de cima para baixo. Se vocês pularem uma célula ou voltarem para editar uma anterior, é preciso rodar tudo de novo a partir dali — senão variáveis antigas continuam "vivas" e os resultados ficam inconsistentes. Na dúvida, usem o menu **Kernel → Restart & Run All** para garantir que tudo está limpo.

## Parte 0 · Imports

In [2]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
print("Bibliotecas carregadas.")

Bibliotecas carregadas.


---
## Parte 1 · Qualidade dos dados (demonstração)

Um dataset `df_clientes` com sujeira deliberada (seed fixo → reprodutível). Rodem as células desta parte e observem: o objetivo aqui é ver o abstrato dos slides virar código de verdade, não escrever nada ainda.

In [3]:
np.random.seed(42)

nomes = ['Ana Silva','Bruno Costa','Carla Melo','Diego Rocha','Elena Prado',
         'Felipe Lima','Gabi Nunes','Hugo Alves','Iara Souza','João Faria',
         'Karla Dias','Lucas Ramos','Mariana Leal','Neto Barros','Olga Mendes',
         'Ana Silva', 'Carlos Melo', 'Carlos M. Melo']  # as 3 últimas são duplicatas/quase-duplicatas

sexo_raw = ['F','F','M','M','F','M','F','Masculino','F','M','F','Masc','F','M','F','F','M','M']

escolaridade = ['Médio','Superior','Superior','Médio','Pós-grad','Médio','Superior',
                'Fundamental','Superior','Pós-grad','Médio','Superior','Pós-grad',
                'Médio','Superior','Médio','Superior','Superior']
esc_num = {'Fundamental': 1, 'Médio': 2, 'Superior': 3, 'Pós-grad': 4}

renda = np.array([esc_num[e] * 1800 + np.random.randint(-300, 300) for e in escolaridade], dtype=float)
for i in [4, 9, 12]:       # MNAR: Pós-grad tende a não declarar renda
    renda[i] = np.nan
renda[2] = -350.0          # validade: renda negativa
renda[15] = renda[0]       # duplicata exata compartilha valores
renda[16] = renda[17] = 3200.0

idade = [28, 34, 41, 25, 52, 29, 37, 19, 45, 61, 31, 27, 58, 22, 43, 28, 41, 41]
idade[7] = -8              # validade: idade impossível

cpf = [f'{700000000 + i*7777:09d}' for i in range(15)] + [f'{700000000:09d}', '987654321', '987654321']

df_clientes = pd.DataFrame({
    'cpf': cpf, 'nome': nomes, 'sexo': sexo_raw, 'idade': idade,
    'escolaridade': escolaridade, 'renda': renda,
})
print(f"Shape: {df_clientes.shape}")
df_clientes

Shape: (18, 6)


,cpf,nome,sexo,idade,escolaridade,renda
0,700000000,Ana Silva,F,28,Médio,3402.00
1,700007777,Bruno Costa,F,34,Superior,5535.00
2,700015554,Carla Melo,M,41,Superior,-350.00
3,700023331,Diego Rocha,M,25,Médio,3406.00
4,700031108,Elena Prado,F,52,Pós-grad,NaN
5,700038885,Felipe Lima,M,29,Médio,3320.00
6,700046662,Gabi Nunes,F,37,Superior,5221.00
7,700054439,Hugo Alves,Masculino,-8,Fundamental,1966.00
8,700062216,Iara Souza,F,45,Superior,5314.00
9,700069993,João Faria,M,61,Pós-grad,NaN


### 1.1 · Auditoria dos quatro pilares

In [4]:
# Completude: quantos campos estão em branco?
display(df_clientes.isna().sum().to_frame('n_missing').query('n_missing > 0'))

# Consistência: mesma entidade, grafias diferentes?
print("\nValores de 'sexo':")
display(df_clientes['sexo'].value_counts())

# Unicidade: linhas repetidas?
print(f"\nDuplicatas exatas: {df_clientes.duplicated().sum()}")
display(df_clientes[df_clientes.duplicated(subset='cpf', keep=False)][['cpf', 'nome']])

# Validade: valores fora do domínio?
print("\nValores de validade violada:")
display(df_clientes[(~df_clientes['idade'].between(0, 120)) | (df_clientes['renda'] < 0)][['nome', 'idade', 'renda']])

,n_missing
renda,3



Valores de 'sexo':


sexo
F            9
M            7
Masculino    1
Masc         1
Name: count, dtype: int64


Duplicatas exatas: 1


,cpf,nome
0,700000000,Ana Silva
15,700000000,Ana Silva
16,987654321,Carlos Melo
17,987654321,Carlos M. Melo



Valores de validade violada:


,nome,idade,renda
2,Carla Melo,41,-350.00
7,Hugo Alves,-8,1966.00


**Pergunta antes de rodar a próxima célula:** a ausência de `renda` está concentrada em algum grupo específico (dica: olhem a escolaridade das linhas com `renda` ausente na tabela acima)? Isso sugere MCAR, MAR ou MNAR?

In [5]:
df_clientes['renda_ausente'] = df_clientes['renda'].isna()
display(pd.crosstab(df_clientes['escolaridade'], df_clientes['renda_ausente'], normalize='index')
        .rename(columns={False: 'tem renda', True: 'sem renda'}))
print("\nPós-grad concentra 100% da ausência → suspeita de MNAR.")

renda_ausente,tem renda,sem renda
escolaridade,,
Fundamental,1.00,0.00
Médio,1.00,0.00
Pós-grad,0.00,1.00
Superior,1.00,0.00



Pós-grad concentra 100% da ausência → suspeita de MNAR.


### 1.2 · Imputação: mediana global × KNNImputer

In [6]:
imp_mediana = SimpleImputer(strategy='median')
renda_mediana = imp_mediana.fit_transform(df_clientes[['renda']])

df_clientes['esc_num'] = df_clientes['escolaridade'].map(esc_num)
imp_knn = KNNImputer(n_neighbors=3)
dados_knn = imp_knn.fit_transform(df_clientes[['renda', 'esc_num']])

idx_ausentes = df_clientes[df_clientes['renda'].isna()].index
comparacao = pd.DataFrame({
    'nome': df_clientes.loc[idx_ausentes, 'nome'],
    'escolaridade': df_clientes.loc[idx_ausentes, 'escolaridade'],
    'mediana_global': renda_mediana[idx_ausentes, 0].round(2),
    'knn_imputer': dados_knn[idx_ausentes, 0].round(2),
})
display(comparacao)
print("\nKNNImputer usa a escolaridade como vizinhança: imputa valores mais altos para Pós-grad.")

,nome,escolaridade,mediana_global,knn_imputer
4,Elena Prado,Pós-grad,3402.00,3499.67
9,João Faria,Pós-grad,3402.00,3499.67
12,Mariana Leal,Pós-grad,3402.00,3499.67



KNNImputer usa a escolaridade como vizinhança: imputa valores mais altos para Pós-grad.


### 1.3 · Limpeza básica

In [7]:
mapa_sexo = {'F': 'F', 'M': 'M', 'Masc': 'M', 'Masculino': 'M'}
df_limpo = df_clientes.copy()
df_limpo['sexo'] = df_limpo['sexo'].map(mapa_sexo)
df_limpo = df_limpo.drop_duplicates(subset='cpf')
df_limpo = df_limpo[df_limpo['idade'].between(0, 120)]

print(f"Antes: {len(df_clientes)} linhas  →  Depois da limpeza: {len(df_limpo)} linhas")
print("\nLembrete (regra de ouro, visto na Aula sobre SVM): fit() só no treino, nunca no dataset completo antes do split.")

Antes: 18 linhas  →  Depois da limpeza: 15 linhas

Lembrete (regra de ouro, visto na Aula sobre SVM): fit() só no treino, nunca no dataset completo antes do split.


---
## Parte 2 · Prática guiada: escala, codificação e Pipeline

Daqui em diante, **vocês escrevem código**. Cada etapa tem uma célula com `assert` que confere automaticamente se o resultado está correto — rodem a célula e leiam a mensagem.

Dataset novo, pequeno e didático: `clientes_banco.csv`, para prever se um cliente é **inadimplente** (`0` = não, `1` = sim) a partir de `idade`, `renda` e `plano` (categórico). Carregado do mesmo jeito que vocês vão carregar o `koi_data.csv` no TP1.

In [8]:
clientes_banco = pd.read_csv('clientes_banco.csv')
print(f"Shape: {clientes_banco.shape}  ·  taxa de inadimplência: {clientes_banco['inadimplente'].mean():.0%}")
clientes_banco.head()

Shape: (150, 4)  ·  taxa de inadimplência: 45%


,idade,renda,plano,inadimplente
0,65.00,3675.26,Prata,0
1,22.00,5886.66,Ouro,1
2,43.00,7657.36,Básico,1
3,21.00,4242.15,Básico,1
4,37.00,2876.03,Ouro,1


### Etapa 1 · Observem

`idade` e `renda` estão em escalas bem diferentes. Rodem a célula abaixo (código pronto, só observar) e comparem a acurácia média de um k-NN (validação cruzada com 10 folds, para um resultado estável) antes e depois de padronizar — é o mesmo experimento do slide "O mesmo k-NN, duas escalas diferentes".

In [9]:
from sklearn.model_selection import StratifiedKFold

X_num = clientes_banco[['idade', 'renda']].values
y = clientes_banco['inadimplente'].values
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=7)

acc_bruto = cross_val_score(KNeighborsClassifier(n_neighbors=7), X_num, y, cv=cv).mean()
pipe_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
acc_scaled = cross_val_score(pipe_scaled, X_num, y, cv=cv).mean()

print(f"k-NN sem padronizar : {acc_bruto:.2%}")
print(f"k-NN padronizado    : {acc_scaled:.2%}")
print("\nRenda (ruído, escala grande) domina a distância sem padronizar, exatamente como no slide da Aula.")

k-NN sem padronizar : 47.33%
k-NN padronizado    : 62.00%

Renda (ruído, escala grande) domina a distância sem padronizar, exatamente como no slide da Aula.


### Etapa 2 · Completem

A coluna `plano` é categórica, com 3 valores sem ordem natural (Básico/Prata/Ouro não formam uma escala). Complete o código abaixo para codificá-la com `OneHotEncoder`.

Dica: reveja o slide "OneHotEncoder na prática" se precisar.

In [10]:
encoder = OneHotEncoder()
plano_codificado = encoder.fit_transform(clientes_banco[['plano']])

assert plano_codificado.shape == (len(clientes_banco), 3), \
    f"esperado (150, 3): uma coluna por categoria de plano. Obtido: {plano_codificado.shape}"
print("✅ OneHotEncoder aplicado corretamente! Categorias:", encoder.get_feature_names_out())

✅ OneHotEncoder aplicado corretamente! Categorias: ['plano_Básico' 'plano_Ouro' 'plano_Prata']


### Etapa 3 · Construam

Agora o desafio principal: montem um `Pipeline` completo, que:

1. usa um `ColumnTransformer` para aplicar `StandardScaler` nas colunas numéricas (`idade`, `renda`) e `OneHotEncoder` na coluna categórica (`plano`);
2. encadeia esse `ColumnTransformer` com um classificador (`SVC(kernel='rbf')`);
3. é avaliado com `cross_val_score` usando `cv=5`, exatamente como no TP1.

Revejam os slides "ColumnTransformer: escalar e codificar ao mesmo tempo" e "Juntando tudo: Pipeline + validação cruzada" se precisarem.

In [11]:
colunas_numericas = ['idade', 'renda']
colunas_categoricas = ['plano']

X = clientes_banco[colunas_numericas + colunas_categoricas]
y = clientes_banco['inadimplente']

pre = ColumnTransformer([
    ('num', StandardScaler(), colunas_numericas),
    ('cat', OneHotEncoder(), colunas_categoricas),
])

modelo = make_pipeline(pre, SVC(kernel='rbf'))

scores = cross_val_score(modelo, X, y, cv=5)

assert len(scores) == 5, f"cross_val_score com cv=5 deve devolver 5 valores, obtido {len(scores)}"
assert 0.5 <= scores.mean() <= 1.0, f"acurácia média fora do intervalo esperado: {scores.mean():.3f}"
print(f"✅ Pipeline correto! Acurácia média: {scores.mean():.2%} ± {scores.std():.2%}")

✅ Pipeline correto! Acurácia média: 59.33% ± 3.27%


**Para refletir antes de seguir:** por que é importante que o `StandardScaler` e o `OneHotEncoder` estejam *dentro* do `Pipeline`, em vez de aplicados antes do `cross_val_score`? (resposta no slide "Vazamento de dados", se tiverem dúvida)

---
## Parte 3 · Aplicação ao TP1

Chegou a hora de aplicar exatamente o que vocês praticaram ao dataset de verdade do TP1. Sem gabarito daqui em diante — vocês vão usar isso no trabalho prático.

In [12]:
koi = pd.read_csv('tp1/koi_data.csv')
print(f"Shape: {koi.shape}")
koi.head()

Shape: (5202, 43)


,kepoi_name,koi_disposition,koi_period,koi_impact,koi_duration,koi_depth,koi_ror,koi_srho,koi_prad,koi_sma,koi_incl,koi_teq,koi_insol,koi_dor,koi_max_sngle_ev,koi_max_mult_ev,koi_model_snr,koi_steff,koi_slogg,koi_smet,koi_srad,koi_smass,koi_kepmag,koi_gmag,koi_rmag,koi_imag,koi_zmag,koi_jmag,koi_hmag,koi_kmag,koi_fwm_stat_sig,koi_fwm_sra,koi_fwm_sdec,koi_fwm_srao,koi_fwm_sdeco,koi_fwm_prao,koi_fwm_pdeco,koi_dicco_mra,koi_dicco_mdec,koi_dicco_msky,koi_dikco_mra,koi_dikco_mdec,koi_dikco_msky
0,K00752.01,CONFIRMED,9.49,0.15,2.96,615.80,0.02,3.21,2.26,0.09,89.66,793.00,93.59,24.81,5.14,28.47,35.80,5455.00,4.47,0.14,0.93,0.92,15.35,15.89,15.27,15.11,15.01,14.08,13.75,13.65,0.00,19.46,48.14,0.43,0.94,-0.00,-0.00,-0.01,0.20,0.20,0.08,0.31,0.32
1,K00752.02,CONFIRMED,54.42,0.59,4.51,874.80,0.03,3.02,2.83,0.27,89.57,443.00,9.11,77.90,7.03,20.11,25.80,5455.00,4.47,0.14,0.93,0.92,15.35,15.89,15.27,15.11,15.01,14.08,13.75,13.65,0.00,19.46,48.14,-0.63,1.23,0.00,-0.00,0.39,0.00,0.39,0.49,0.12,0.50
2,K00754.01,FALSE POSITIVE,1.74,1.28,2.41,8079.20,0.39,0.22,33.46,0.03,67.09,1395.00,891.96,3.28,39.07,541.90,505.60,5805.00,4.56,-0.52,0.79,0.84,15.60,16.10,15.55,15.38,15.27,14.33,13.91,13.81,0.00,19.04,48.29,-0.11,0.00,0.00,-0.00,-0.25,0.15,0.29,-0.26,0.10,0.28
3,K00755.01,CONFIRMED,2.53,0.70,1.65,603.30,0.02,1.99,2.75,0.04,85.41,1406.00,926.16,8.75,4.75,33.19,40.90,6031.00,4.44,0.07,1.05,1.09,15.51,16.02,15.47,15.29,15.24,14.37,14.06,13.95,0.73,19.25,48.23,-0.01,0.23,0.00,-0.00,0.03,-0.09,0.10,0.07,0.02,0.07
4,K00114.01,FALSE POSITIVE,7.36,1.17,5.02,233.70,0.18,0.00,39.21,0.08,60.92,1342.00,767.22,2.40,10.96,46.15,47.70,6227.00,3.99,0.00,1.96,1.36,12.66,13.00,12.61,12.52,12.48,11.66,11.41,11.40,0.00,19.92,42.16,-13.45,24.09,0.00,-0.01,-4.51,7.71,8.93,-4.54,7.71,8.95


Reparem: diferente do `clientes_banco`, o `koi_data.csv` **não tem nenhuma coluna categórica** (todas as features, exceto o identificador e o rótulo, são numéricas) e **não tem valores ausentes**. Ou seja, o `ColumnTransformer` com duas ramificações (numérica + categórica) não é necessário aqui — um `Pipeline` só com `StandardScaler` já resolve a parte de pré-processamento.

**Tarefa (sem assert, sem gabarito):**

1. Separem `X` (as colunas de features, do índice 2 em diante) e `y` (a coluna `koi_disposition`, convertida para 0/1).
2. Montem um `Pipeline` com `StandardScaler` + um classificador da lista do TP1 (comecem com `SVC` ou `KNeighborsClassifier`, que são os mais sensíveis a escala).
3. Rodem `cross_val_score` com `cv=5` e comparem a acurácia com e sem o `StandardScaler` no Pipeline.
4. Guardem esse código: ele é o ponto de partida do TP1.

In [13]:
X_koi = koi.iloc[:, 2:]
y_koi = (koi['koi_disposition'] == 'CONFIRMED').astype(int)

print(f"X: {X_koi.shape}  ·  y: {y_koi.shape}  ·  taxa de CONFIRMED: {y_koi.mean():.1%}")

pipe_raw = make_pipeline(KNeighborsClassifier(n_neighbors=7))
pipe_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))

acc_raw = cross_val_score(pipe_raw, X_koi, y_koi, cv=5).mean()
acc_scaled = cross_val_score(pipe_scaled, X_koi, y_koi, cv=5).mean()

print(f"k-NN sem StandardScaler : {acc_raw:.2%}")
print(f"k-NN com StandardScaler : {acc_scaled:.2%}")

pipe_svc = make_pipeline(StandardScaler(), SVC(kernel='rbf'))
acc_svc = cross_val_score(pipe_svc, X_koi, y_koi, cv=5).mean()
print(f"SVC(rbf) com StandardScaler: {acc_svc:.2%}")

X: (5202, 41)  ·  y: (5202,)  ·  taxa de CONFIRMED: 40.4%
k-NN sem StandardScaler : 77.76%
k-NN com StandardScaler : 88.26%
SVC(rbf) com StandardScaler: 93.46%


---
## Fechamento

Depois de terminar a Parte 2 e explorar a Parte 3, vocês devem ser capazes de responder:

- Por que `idade` e `renda` em escalas diferentes distorcem um k-NN ou SVM?
- Por que `OneHotEncoder` é mais seguro que `LabelEncoder` para uma feature categórica sem ordem?
- Por que o `StandardScaler`/`OneHotEncoder` precisa estar *dentro* do `Pipeline`, e não aplicado antes do `cross_val_score`?

Se alguma dessas perguntas ainda estiver confusa, é hora de rever o slide correspondente antes de começar o TP1.